COMP34212 Coursework
=========


**Importing the libraries**

In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import SGD, Adam, RMSprop
from tensorflow.keras.callbacks import ModelCheckpoint

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print('Libraries imported.')

Libraries imported.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Initialising Base Variables and Constants**

These will be the default variables and constants.

In [ ]:
BATCH_SIZE = 128
N_EPOCH = 20
N_CLASSES = 10
VERBOSE = 1
VALIDATION_SPLIT = 0.2
LEARNING_RATE = 0.001

IMG_CHANNELS = 3
IMG_ROWS = 32
IMG_COLS = 32

print('Variables and constants initialised.')

Variables and constants initialised.


**Experimental Hyperparameters**

I will be experimenting with the following hyperparameters and seeing the results.

In [ ]:
batch_sizes = [64, 128, 256]
epochs_list = [10, 20]
learning_rates = [0.001, 0.01]
dropout_rates = [0.25, 0.5]

print('Experimental hyperparameters initialised.')

Experimental hyperparameters initialised.


__Loading and Processing CIFAR-10 Data__

We will be working with the CIFAR-10 dataset for this exercise

In [ ]:
#load dataset
(input_X_train, output_y_train), (input_X_test, output_y_test) = cifar10.load_data()
print('input_X_train shape:', input_X_train.shape)
print(input_X_train.shape[0], 'train samples')
print(input_X_test.shape[0], 'test samples')

# convert to categorical
output_Y_train = utils.to_categorical(output_y_train, N_CLASSES)
output_Y_test = utils.to_categorical(output_y_test, N_CLASSES)

# float and normalization
input_X_train = input_X_train.astype('float32')
input_X_test = input_X_test.astype('float32')
input_X_train /= 255
input_X_test /= 255


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
input_X_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples


**Training the CNN**

Here, we'll train the model with the different hyperparameter configurations. For this I'll be using a simple CNN model like the one use in one of the labs.


In [ ]:
results = {}
for batch_size in batch_sizes:
  for epochs in epochs_list:
    for learning_rate in learning_rates:
      for dropout_rate in dropout_rates:

        print("Hyperparameters used:")
        print(f"Batch size: {batch_size}")
        print(f"Epochs: {epochs}")
        print(f"Learning rate: {learning_rate}")
        print(f"Dropout rate: {dropout_rate}")

        model = Sequential()
        model.add(Conv2D(32, (3, 3), padding='same', input_shape=(IMG_ROWS, IMG_COLS, IMG_CHANNELS)))
        model.add(Activation('relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

        model.add(Flatten())
        model.add(Dense(512))
        model.add(Activation('relu'))
        model.add(Dropout(0.5))

        model.add(Dense(N_CLASSES))
        model.add(Activation('softmax'))

        OPTIM1 = RMSprop(learning_rate=learning_rate)
        model.compile(loss='categorical_crossentropy', optimizer=OPTIM1, metrics=['accuracy'])

        history = model.fit(input_X_train, output_Y_train, batch_size=batch_size, epochs=epochs, validation_split=VALIDATION_SPLIT,  verbose=VERBOSE)

        results[(batch_size, epochs, learning_rate, dropout_rate)] = {
                    'accuracy': history.history['accuracy'],
                    'val_accuracy': history.history['val_accuracy'],
                    'loss': history.history['loss'],
                    'val_loss': history.history['val_loss']
                }

Hyperparameters used:
Batch size: 64
Epochs: 10
Learning rate: 0.001
Dropout rate: 0.25


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.2987 - loss: 2.0110 - val_accuracy: 0.5370 - val_loss: 1.3376
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5123 - loss: 1.3815 - val_accuracy: 0.5775 - val_loss: 1.2132
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5649 - loss: 1.2257 - val_accuracy: 0.6088 - val_loss: 1.1254
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6036 - loss: 1.1160 - val_accuracy: 0.6295 - val_loss: 1.0578
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6314 - loss: 1.0469 - val_accuracy: 0.6465 - val_loss: 1.0317
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6553 - loss: 0.9931 - val_accuracy: 0.6583 - val_loss: 1.0221
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6780 - loss: 0.9300 - val_accuracy: 0.6439 - val_loss: 1.0538
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.6857 - loss: 0.8927 - val_accuracy: 0.

**Results Table**

The following gives us a table of the best hyperparameter configurations, and we can see that with what has the highest val_accuracy



In [ ]:
results_df = pd.DataFrame.from_dict(results, orient='index', columns=['accuracy', 'val_accuracy', 'loss', 'val_loss'])

results_df['batch_size'] = [key[0] for key in results_df.index]
results_df['epochs'] = [key[1] for key in results_df.index]
results_df['learning_rate'] = [key[2] for key in results_df.index]
results_df['dropout_rate'] = [key[3] for key in results_df.index]

results_df = results_df.reset_index()

results_df['accuracy'] = [history['accuracy'][-1] for history in results.values()]
results_df['val_accuracy'] = [history['val_accuracy'][-1] for history in results.values()]
results_df['loss'] = [history['loss'][-1] for history in results.values()]
results_df['val_loss'] = [history['val_loss'][-1] for history in results.values()]

results_df = results_df[['batch_size', 'epochs', 'learning_rate', 'dropout_rate', 'accuracy', 'val_accuracy', 'loss', 'val_loss']]

results_df_sorted = results_df.sort_values(by='val_accuracy', ascending=False)

print("Best hyperparameter configurations:")
results_df_sorted

Best hyperparameter configurations:


,batch_size,epochs,learning_rate,dropout_rate,accuracy,val_accuracy,loss,val_loss
12,128,20,0.001,0.25,0.802050,0.6824,0.566236,1.011089
5,64,20,0.001,0.50,0.739825,0.6820,0.769720,0.987905
13,128,20,0.001,0.50,0.731575,0.6818,0.770228,0.965160
20,256,20,0.001,0.25,0.768025,0.6692,0.664803,0.974773
0,64,10,0.001,0.25,0.720125,0.6668,0.811378,0.990701
4,64,20,0.001,0.25,0.828650,0.6634,0.502863,1.095483
21,256,20,0.001,0.50,0.711050,0.6557,0.823470,0.996205
1,64,10,0.001,0.50,0.668375,0.6530,0.966164,1.032257
9,128,10,0.001,0.50,0.672800,0.6469,0.931830,1.022421
8,128,10,0.001,0.25,0.679775,0.6353,0.908446,1.063868


**Graphs of the Results**

The following displays a total of 48 graphs, 2 for each configurations of which there are 24.

In [ ]:
for (batch_size, epochs, learning_rate, dropout_rate), history in results.items():
  print("Hyperparameters:")
  print(f"Batch size: {batch_size}")
  print(f"Epochs: {epochs}")
  print(f"Learning rate: {learning_rate}")
  print(f"Dropout rate: {dropout_rate}")

  score = model.evaluate(input_X_test, output_Y_test, batch_size=BATCH_SIZE, verbose=VERBOSE)
  print("\nTest score/loss:", score[0])
  print('Test accuracy:', score[1])

  plt.figure(figsize=(10, 5))
  plt.plot(history['accuracy'])
  plt.plot(history['val_accuracy'])
  plt.title(f'Model Accuracy for: Batch Size={batch_size}, Epochs={epochs}, Learning Rate={learning_rate}, Dropout Rate={dropout_rate}')
  plt.ylabel('Accuracy')
  plt.xlabel('Epoch')
  plt.legend(['train', 'test'], loc='upper left')
  plt.savefig(f'./drive/MyDrive/Colab Notebooks/34212 Labs/graphs/accuracy_{batch_size}_{epochs}_{learning_rate}_{dropout_rate}.png')
  plt.show()

  plt.figure(figsize=(10, 5))
  plt.plot(history['loss'])
  plt.plot(history['val_loss'])
  plt.title(f'Model Loss for: Batch Size={batch_size}, Epochs={epochs}, Learning Rate={learning_rate}, Dropout Rate={dropout_rate}')
  plt.ylabel('Loss')
  plt.xlabel('Epoch')
  plt.legend(['train', 'test'], loc='upper left')
  plt.savefig(f'./drive/MyDrive/Colab Notebooks/34212 Labs/graphs/loss_{batch_size}_{epochs}_{learning_rate}_{dropout_rate}.png')
  plt.show()


Output hidden; open in https://colab.research.google.com to view.